# TMJDLC — Javtiful Telegram Bot

Google Colab runner for TMJDLC. The private GitHub data repository stores bot credentials, settings, the durable SQLite queue snapshot, channel catalogue, and completed upload records. Colab needs only `GITHUB_TOKEN`.

Pipeline supports up to **2 download workers + 2 upload workers**, with a bounded upload backlog. Run cells in order. Keep the bot cell running while processing jobs.


## 1. GitHub token

TMJDLC accepts `GITHUB_TOKEN` from the Colab runtime environment (including Google Colab CLI environment injection). If it is not present, the notebook falls back to Colab Secrets. The token needs **Contents: Read and write** access to the private `gitroff/Database` repository.

Do not put the token in this notebook or in the public TMJDLC repository.


In [ ]:
#@title 2. Sync latest TMJDLC source
%cd /content
import os, subprocess
repo = '/content/TMJDLC'
if os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', '-C', repo, 'fetch', '--prune', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', repo, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', repo, 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', 'https://github.com/Telebotfaroff/TMJDLC.git', repo], check=True)
%cd /content/TMJDLC
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'✅ TMJDLC source synced to main: {commit}')
subprocess.run(['git', 'status', '--short'], check=True)


In [ ]:
#@title 3. Install dependencies (run once per fresh Colab runtime)
%cd /content/TMJDLC
import subprocess, sys
print('Installing TMJDLC and Python dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
print('Installing FFmpeg and aria2...')
subprocess.run(['bash', '-lc', 'apt-get update -qq && apt-get install -y ffmpeg aria2'], check=True)
print('Installing Playwright Chromium and system dependencies...')
subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'], check=True)
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'tmjdlc'], check=True)
print('✅ Dependencies installed and Python modules compiled.')


In [ ]:
#@title 4. Load the GitHub data-repository token
import os

# Colab CLI injects environment variables directly; regular Colab can use Secrets.
github_token = os.environ.get('GITHUB_TOKEN', '').strip()
if not github_token:
    try:
        from google.colab import userdata
        github_token = str(userdata.get('GITHUB_TOKEN') or '').strip()
    except Exception:
        github_token = ''
if not github_token:
    raise RuntimeError('GITHUB_TOKEN is missing. Inject it through Colab CLI or add it to Colab Secrets and enable notebook access.')
os.environ['GITHUB_TOKEN'] = github_token
print('✅ GitHub data-repository token loaded. Telegram credentials will be read from the private repository.')


In [ ]:
#@title 5. Preflight checks
import shutil, importlib.util, sys, os
from pathlib import Path
if sys.version_info < (3, 12):
    raise RuntimeError(f'TMJDLC requires Python 3.12+. Current runtime: {sys.version.split()[0]}')
repo_dir = '/content/TMJDLC'
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
importlib.invalidate_caches()
if not os.environ.get('GITHUB_TOKEN', '').strip():
    raise RuntimeError('GITHUB_TOKEN is not loaded. Run cell 4 successfully before starting TMJDLC.')
for executable in ('ffmpeg', 'ffprobe', 'aria2c'):
    if shutil.which(executable) is None:
        raise RuntimeError(f'{executable} is not installed or not on PATH.')
for module in ('pyrogram', 'playwright', 'requests', 'tmjdlc'):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError(f'Required Python module is missing: {module}. Re-run dependency installation.')
work_dir = Path(os.environ.get('TMJDLC_WORK_DIR', '/content/tmjdlc-bot'))
work_dir.mkdir(parents=True, exist_ok=True)
disk = shutil.disk_usage(work_dir)
print(f'Python: {sys.version.split()[0]} | FFmpeg/FFprobe/aria2: ready')
print(f'Free Colab disk: {disk.free / 1024**3:.1f} GiB')
print('✅ Preflight passed. GitHub configuration and database recovery run at bot startup.')


## 6. Start the bot

Run the next cell and keep it running. Send `/start` to the main bot. The bot reads Telegram credentials and settings from the private GitHub data repository.

The durable queue uses local SQLite for fast transactional worker coordination and synchronizes snapshots to GitHub. Completed upload records and the channel catalogue are also saved in the repository. Colab media files remain temporary.


### Colab Free: closing the browser\n\nColab Free does not provide an always-on runtime. If you close the browser or switch off your device, the running VM may continue for some time, but Google can disconnect or reclaim it at any time. **There is no supported setting or notebook cell that guarantees the bot will keep running after the browser is closed.**\n\n- Keep the bot start cell running; do not click **Stop**, **Disconnect and delete runtime**, or interrupt execution while jobs are active.\n- Closing the browser is not the same as stopping the runtime, but it is not a persistence guarantee.\n- If the runtime disconnects, open this notebook again, reconnect, and rerun the setup cells as needed. The GitHub-backed snapshots restore saved queue state; files stored only on Colab's temporary disk may need to be downloaded again.\n- Do not run two Colab sessions against the same data repository at the same time.\n

In [ ]:
#@title 7. Start TMJDLC — WZGram MTProto
%cd /content/TMJDLC
import os, sys, subprocess
from pathlib import Path
repo_dir = '/content/TMJDLC'
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
if not os.environ.get('GITHUB_TOKEN', '').strip():
    raise RuntimeError('Start blocked: GITHUB_TOKEN is missing. Complete cell 4 first.')
import tmjdlc
work_dir = Path(os.environ.get('TMJDLC_WORK_DIR', '/content/tmjdlc-bot'))
work_dir.mkdir(parents=True, exist_ok=True)
log_path = work_dir / 'bot.log'
print('Starting TMJDLC. Keep this cell running while the bot is in use.')
command = f'PYTHONUNBUFFERED=1 "{sys.executable}" -u -m tmjdlc.bot 2>&1 | tee -a "{log_path}"'
subprocess.run(['bash', '-o', 'pipefail', '-c', command], check=True)


## Telegram controls

- `/start`, `/panel` — dashboard
- **Download Video** — queue one Javtiful video URL
- **Batch Download** — inspect a channel and choose how many videos to queue
- `/channels` — scan Javtiful for channel URLs and browse the saved catalogue
- `/commands` — hide/show, rename, and reorder menu buttons
- **Queue Manager** — pause, resume, cancel, retry, and inspect jobs
- **Settings** — configure 1–2 download workers, 1–2 upload workers, and 0–2 waiting uploads per worker
- `/processes` — live download/upload monitor with worker IDs, progress, and upload backlog
- `/storage`, `/cancel JOB_ID`, `/retry JOB_ID` — runtime and job controls

### Pipeline and storage

Configure worker counts and backlog from Telegram Settings. The queue pauses new downloads when the active upload slots or the configured ready-to-upload backlog are full. The hard maximum is two workers in each pool. Completed Telegram and GoFile receipts are saved as individual JSON records in the private GitHub data repository.
